# ESM Embedding Exploration

*Write intro here explaining purpose

In [18]:
from transformers import AutoTokenizer, TFEsmModel 
import tensorflow as tf
from Bio import SeqIO
import sklearn
import plotly.express as px

hug_face_id = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(hug_face_id)
model = TFEsmModel.from_pretrained(hug_face_id)

# sanity check
inputs = tokenizer("Hello, my dog is cute", return_tensors="tf")
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state

print(last_hidden_states, outputs)

/home/ek224/anaconda3/envs/hsg/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFEsmModel: ['esm.embeddings.position_ids', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing TFEsmModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFEsmModel from a PyTorch model that you expect to be exactly identical (e.g.

tf.Tensor(
[[[ 0.05059963  0.10085721 -0.10023749 ... -0.12190716  0.40376827
    0.08775394]
  [ 0.26290226  0.02058962 -0.16462716 ... -0.05984128  0.3401519
    0.15950891]
  [ 0.13614741  0.17050998  0.10192016 ... -0.17278999  0.32610363
    0.48546052]
  ...
  [ 0.28022113  0.06663495 -0.09451015 ... -0.11177716  0.1806937
    0.5831055 ]
  [ 0.19588457  0.14026381  0.07719216 ... -0.14424258  0.22779934
    0.39744595]
  [ 0.5040827   0.16508391 -0.00226392 ... -0.24705702  0.12229864
    0.42984688]]], shape=(1, 8, 1280), dtype=float32) TFBaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=<tf.Tensor: shape=(1, 8, 1280), dtype=float32, numpy=
array([[[ 0.05059963,  0.10085721, -0.10023749, ..., -0.12190716,
          0.40376827,  0.08775394],
        [ 0.26290226,  0.02058962, -0.16462716, ..., -0.05984128,
          0.3401519 ,  0.15950891],
        [ 0.13614741,  0.17050998,  0.10192016, ..., -0.17278999,
          0.32610363,  0.48546052],
        ...,
        [ 0

In [19]:
# load heme variant sequences
data_path = "../data/Human_Hb_variants.fasta"
variant_entry = []
with open(data_path, "r") as file:
    for seq in SeqIO.parse(file, "fasta"):
        variant_entry.append(seq)

print("Sample Output:")
print(variant_entry[0].id)
print(variant_entry[0].seq)
print("Number of entries:")
print(len(variant_entry))


Sample Output:
alpha1
MGLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
Number of entries:
1449


In [20]:
# check duplicates / tag their indices
seqs = []
duplicates = []
for i, entry in enumerate(variant_entry):
    sequence = str(entry.seq)
    duplicates = []
    if sequence not in seqs:
        seqs.append(sequence)
    else:
        duplicates.append(i)
        print(f"Duplicate sequence found at index {i}")

print(f"Number of duplicates: {len(duplicates)}")
seqs = list(set(seqs))
print(f"Number of unique sequences: {len(seqs)}")

Duplicate sequence found at index 1
Duplicate sequence found at index 3
Duplicate sequence found at index 5
Duplicate sequence found at index 7
Duplicate sequence found at index 9
Duplicate sequence found at index 11
Duplicate sequence found at index 13
Duplicate sequence found at index 15
Duplicate sequence found at index 17
Duplicate sequence found at index 19
Duplicate sequence found at index 21
Duplicate sequence found at index 23
Duplicate sequence found at index 25
Duplicate sequence found at index 27
Duplicate sequence found at index 29
Duplicate sequence found at index 31
Duplicate sequence found at index 34
Duplicate sequence found at index 37
Duplicate sequence found at index 39
Duplicate sequence found at index 41
Duplicate sequence found at index 43
Duplicate sequence found at index 45
Duplicate sequence found at index 47
Duplicate sequence found at index 49
Duplicate sequence found at index 52
Duplicate sequence found at index 54
Duplicate sequence found at index 56
Duplic

In [21]:
print(duplicates)

[]


In [22]:
print(set([variant_entry[i].id for i,entry in enumerate(variant_entry)]))

{'beta', 'Agamma', '744', '746', '2807', '747', '745', '743', '750', 'Ggamma', 'delta', '751', '3057', 'alpha2', 'alpha1', '742'}


Marengo-Rowe AJ. Structure-function relations of human hemoglobins. Proc (Bayl Univ Med Cent). 2006 Jul;19(3):239-45. doi: 10.1080/08998280.2006.11928171. PMID: 17252042; PMCID: PMC1484532.

"The alpha chain of all human hemoglobins, embryonic and adult, is the same. The non-alpha chains include the beta chain of normal adult hemoglobin (α2β2), the gamma chain of fetal hemoglobin (α2β2), and the delta chain of HbA2. In some variants, the gamma genes are duplicated, giving rise to two kinds of gamma chains."

In [23]:
binned_seqs = {'2807':[], 
               '751':[], 
               '745':[],
               '750':[], 
               '747':[],
               '3057':[], 
               'Agamma':[], 
               '746':[], 
               '743':[], 
               'beta':[], 
               'alpha1':[], 
               '742':[], 
               '744':[], 
               'delta':[], 
               'Ggamma':[], 
               'alpha2':[]}

for i in variant_entry:
    binned_seqs[str(i.id)].append(str(i.seq))

for x in binned_seqs:
    print(f"Number of sequences in {x}: {len(binned_seqs[x])}")

Number of sequences in 2807: 1
Number of sequences in 751: 1
Number of sequences in 745: 2
Number of sequences in 750: 1
Number of sequences in 747: 1
Number of sequences in 3057: 2
Number of sequences in Agamma: 42
Number of sequences in 746: 1
Number of sequences in 743: 1
Number of sequences in beta: 563
Number of sequences in alpha1: 295
Number of sequences in 742: 1
Number of sequences in 744: 1
Number of sequences in delta: 91
Number of sequences in Ggamma: 72
Number of sequences in alpha2: 374


Treating Numerical Entries as aberrent, cluster variant embeddings by chain...

In [24]:
var_embed_dict = {
    "Agamma": list(set(binned_seqs['Agamma'])),
    "Ggamma": list(set(binned_seqs['Ggamma'])),
    "alpha1": list(set(binned_seqs['alpha1'])),
    "alpha2": list(set(binned_seqs['alpha2'])),
    "beta": list(set(binned_seqs['beta'])),
    "delta": list(set(binned_seqs['delta'])),
}
del binned_seqs

for i in var_embed_dict:
    print(f"Number of sequences in {i}: {len(var_embed_dict[i])}")

Number of sequences in Agamma: 42
Number of sequences in Ggamma: 72
Number of sequences in alpha1: 295
Number of sequences in alpha2: 374
Number of sequences in beta: 563
Number of sequences in delta: 91


In [ ]:
import os
import numpy as np

output_directory = "../data/variant_embeddings/"
os.makedirs(output_directory, exist_ok=True)

max_len = max([len(s) for s in seqs])
print(f"Max sequence length: {max_len}")

for i in var_embed_dict:
    print(f"Processing {i}")
    with open(output_directory + f"{i}.csv", "wb") as file:
        count = 0
        for j in var_embed_dict[i]:
            input = tokenizer(j, return_tensors="tf", padding=True, truncation=True, max_length=max_len)
            output = model(**input)
            np.save(file, np.array(output.last_hidden_state))
            count += 1
        print(f"Number of sequences processed: {count}")
    print(f"----- Embeddings for {i} saved -----")

#for i in var_embed_dict:
#    print(f"Processing {i}")
#    inputs = tokenizer(var_embed_dict[i], return_tensors="tf", padding=True, truncation=True, max_length=max_len)
#    print("Generating embeddings")
#    outputs = model(inputs)
#    print("Saving embeddings")
#    os.makedirs(output_directory, exist_ok=True)
#    with open(output_directory + f"{i}.csv", "w") as file:
#        writer = csv.writer(file).writerows(np.array(outputs.last_hidden_state))
#    print(f"----- Embeddings for {i} saved -----")

Max sequence length: 148
Processing Agamma
Number of sequences processed: 42
----- Embeddings for Agamma saved -----
Processing Ggamma
Number of sequences processed: 72
----- Embeddings for Ggamma saved -----
Processing alpha1
Number of sequences processed: 295
----- Embeddings for alpha1 saved -----
Processing alpha2
Number of sequences processed: 374
----- Embeddings for alpha2 saved -----
Processing beta
Number of sequences processed: 563
----- Embeddings for beta saved -----
Processing delta
Number of sequences processed: 91
----- Embeddings for delta saved -----


# Plot UMAPs of Embeddings

In [26]:
# load embeddings
Agamma = np.loadtxt(output_directory + "Agamma.csv", delimiter=",")
Ggamma = np.loadtxt(output_directory + "Ggamma.csv", delimiter=",")
alpha1 = np.loadtxt(output_directory + "alpha1.csv", delimiter=",")
alpha2 = np.loadtxt(output_directory + "alpha2.csv", delimiter=",")
beta = np.loadtxt(output_directory + "beta.csv", delimiter=",")
delta = np.loadtxt(output_directory + "delta.csv", delimiter=",")
print(Agamma.shape, Ggamma.shape, alpha1.shape, alpha2.shape, beta.shape, delta.shape)

ValueError: could not convert string '"[ 0.04951056 -0.01048073  0.01872822 ... -0.23288116  0.15158829' to float64 at row 0, column 1.